In [0]:
import pandas as pd

In [0]:
spark.range(1).show()

+---+
| id|
+---+
|  0|
+---+



In [0]:
%sql
SELECT current_user() AS usuario, 
       current_catalog() AS catalogo,
       current_schema() AS esquema


usuario,catalogo,esquema
carmenmarin689@gmail.com,workspace,default


In [0]:
%sql
SHOW CATALOGS

catalog
analitica
samples
system
workspace


# 1. Creación del entorno

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS analitica

In [0]:
%sql
USE CATALOG analitica


## 1.1 Creación del esquema dentro de nuestro catalogo


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS analitica.data;
USE SCHEMA data;

## 1.2 Creación del Volumen


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS analitica.data.landing


In [0]:
%sql
SHOW VOLUMES IN analitica.data


database,volume_name
data,landing


# 2. Creación de tablas con SQL

In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple (
  id INT,
  nombre STRING
)
USING DELTA

## 2.1 Insertar datos en la tabla

In [0]:
%sql
INSERT INTO tabla_simple VALUES (1, 'Ana'), (2, 'Luis'), (3, 'María')


num_affected_rows,num_inserted_rows
3,3


##2.2 Mostrar datos

In [0]:
%sql
SELECT * FROM tabla_simple ORDER BY id


id,nombre
1,Ana
2,Luis
3,María


# 3. Ruta y funciones de PySpark

Este es una prueba de como puede quedar la ruta

In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/analitica/data/landing"

##3.1 Crear funciones

In [0]:
%sql
CREATE OR REPLACE FUNCTION analitica.data.fn_mayor_que_10(x DOUBLE)
RETURNS boolean
RETURN x > 10

In [0]:
%sql
SELECT * FROM tabla_simple
WHERE analitica.data.fn_mayor_que_10(2)

id,nombre


# 4. Lectura de datos 

También se puede usar ruta + "/data_demo.csv"  

In [0]:
path_data_demo = "/Volumes/analitica/data/landing/data_demo.csv"

Podemos usar el .limit en el display para que nos muestre los primeros que yo quiera 10 o 5 

In [0]:
df_inmuebles = spark.read.csv(
    path= path_data_demo, header=True, inferSchema=True
)

display(df_inmuebles.limit(10))

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0
3.0,3.0,198.0,3116290.0,3,306.3557294709182,0
4.0,2.0,245.0,1202560.0,3,24.824464122611424,0
2.0,2.0,66.0,419282.0,5,24.0,0
2.0,1.0,114.0,1830260.0,1,251.39533835163672,0
2.0,1.0,56.0,274005.0,5,93.0,0


In [0]:
df_inmuebles.printSchema()
print("Inmuebles:", df_inmuebles.count())

root
 |-- rooms: double (nullable = true)
 |-- garages: double (nullable = true)
 |-- useful_area: double (nullable = true)
 |-- value: double (nullable = true)
 |-- interior_quality: integer (nullable = true)
 |-- time_on_market: double (nullable = true)
 |-- has_outlier: integer (nullable = true)

Inmuebles: 8969


## 4.1 Guardar Tablas

In [0]:
df_inmuebles.write.format("delta").mode("overwrite").saveAsTable("inmuebles")


In [0]:
%sql
SELECT rooms, garages, useful_area, value FROM inmuebles LIMIT 10


rooms,garages,useful_area,value
3.0,1.0,105.0,1038640.0
3.0,2.0,76.0,606405.0
3.0,1.0,123.0,1534500.0
3.0,2.0,180.0,1131950.0
3.0,1.0,67.0,452672.0
3.0,3.0,198.0,3116290.0
4.0,2.0,245.0,1202560.0
2.0,2.0,66.0,419282.0
2.0,1.0,114.0,1830260.0
2.0,1.0,56.0,274005.0


#`Exploración con Spark y operaciones Delta`

# 5. Lectura Pedidos 
Una fila de spark_intro.csv representa un pedido del ejemplo. El archivo contiene diez filas de enero de 2024 y no declara moneda. Usamos tipos explícitos para que el esquema sea fácil de revisar.



In [0]:
path_pedidos = "/Volumes/analitica/data/landing/spark_intro.csv"

In [0]:
df_pedidos = spark.read.csv(
    path= path_pedidos, header=True,
    schema="""order_id INT, order_date DATE, country STRING, product STRING,
              category STRING, units_sold INT, unit_price DOUBLE""",
    dateFormat="yyyy-MM-dd", mode="FAILFAST"
)

display(df_pedidos)

order_id,order_date,country,product,category,units_sold,unit_price
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0
6,2024-01-08,France,Mouse,Accessories,10,23.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0
9,2024-01-12,France,Monitor,Electronics,1,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0


## 5.1 Selección y filtro
Elegimos columnas y conservamos los pedidos de Spain. El filtro devuelve las filas que cumplen la condición y deja intacto el DataFrame original.

In [0]:
display(
    df_pedidos.select("order_id", "country", "product", "units_sold")
    .filter(F.col("country") == "Spain")
)

order_id,country,product,units_sold
1,Spain,Laptop,2
2,Spain,Mouse,5
5,Spain,Monitor,2
8,Spain,Keyboard,4


## 5.2 Columna calculada
Para este ejercicio interpretamos unit_price como precio por unidad. El importe de cada pedido es units_sold por unit_price.

In [0]:
df_importes = df_pedidos.withColumn(
    "importe", F.col("units_sold") * F.col("unit_price")
)

display(df_importes)

order_id,order_date,country,product,category,units_sold,unit_price,importe
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0,2400.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0,125.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0,135.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0,600.0
6,2024-01-08,France,Mouse,Accessories,10,23.0,230.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0,168.0
9,2024-01-12,France,Monitor,Electronics,1,280.0,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0,162.0


## 5.3 Agrupación
Agrupamos por país para sumar unidades e importes. Cada fila del resultado corresponde a un país.

In [0]:
resumen_pais = df_importes.groupBy("country").agg(
    F.sum("units_sold").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("importe_total")
)

display(resumen_pais.orderBy(F.desc("importe_total")))

country,unidades,importe_total
Spain,13,3293.0
France,12,1660.0
Germany,10,1597.0


# 6. Lectura de la dimensión de productos 
dim_spark_intro.csv contiene cuatro productos, su familia y su proveedor. La llave de la dimensión es product.



In [0]:
path_productos = "/Volumes/analitica/data/landing/dim_spark_intro.csv"

In [0]:
df_productos = spark.read.csv(
    path=path_productos, header=True,
    schema="product STRING, product_family STRING, supplier STRING",
    mode="FAILFAST"
)

display(df_productos)

product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


## 6.1 Unión sencilla y comprobación
Primero revisamos que la dimensión tenga una fila por producto. Después hacemos una unión izquierda. Esperamos conservar los diez pedidos y no encontrar productos sin correspondencia.



In [0]:
display(df_productos.groupBy("product").count().filter("count > 1"))

df_enriquecido = df_importes.join(df_productos, "product", "left")

display(df_enriquecido)
print("Pedidos antes:", df_pedidos.count())
print("Pedidos después:", df_enriquecido.count())
display(df_pedidos.join(df_productos, "product", "left_anti"))

assert df_enriquecido.count() == df_pedidos.count(), "Revisar la llave de productos."

product,count


product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200.0,2400.0,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25.0,125.0,Peripherals,InputMakers
Laptop,3,2024-01-05,France,Electronics,1,1150.0,1150.0,Computers,TechCorp
Keyboard,4,2024-01-06,Germany,Accessories,3,45.0,135.0,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300.0,600.0,Displays,ScreenWorld
Mouse,6,2024-01-08,France,Accessories,10,23.0,230.0,Peripherals,InputMakers
Laptop,7,2024-01-10,Germany,Electronics,1,1300.0,1300.0,Computers,TechCorp
Keyboard,8,2024-01-11,Spain,Accessories,4,42.0,168.0,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280.0,280.0,Displays,ScreenWorld
Mouse,10,2024-01-14,Germany,Accessories,6,27.0,162.0,Peripherals,InputMakers


Pedidos antes: 10
Pedidos después: 10


product,order_id,order_date,country,category,units_sold,unit_price


## 6.2 Guardar y volver a consultar
Guardamos este resultado como tabla Delta. La tabla persistente puede consultarse por nombre desde otro notebook con los permisos necesarios.



In [0]:
df_enriquecido.write.format("delta").mode("overwrite").saveAsTable("pedidos_enriquecidos")


In [0]:
%sql
SELECT product, product_family, supplier, importe FROM pedidos_enriquecidos ORDER BY order_id


product,product_family,supplier,importe
Laptop,Computers,TechCorp,2400.0
Mouse,Peripherals,InputMakers,125.0
Laptop,Computers,TechCorp,1150.0
Keyboard,Peripherals,InputMakers,135.0
Monitor,Displays,ScreenWorld,600.0
Mouse,Peripherals,InputMakers,230.0
Laptop,Computers,TechCorp,1300.0
Keyboard,Peripherals,InputMakers,168.0
Monitor,Displays,ScreenWorld,280.0
Mouse,Peripherals,InputMakers,162.0


# 7. Lectura de ventas
Ahora usamos ventas_2025.csv, que contiene cien filas. Para el ejercicio, cantidad representa unidades y precio el precio unitario. El archivo no declara moneda. Cada id identifica un registro de venta.



In [0]:
path_ventas = "/Volumes/analitica/data/landing/ventas_2025.csv"

In [0]:
df_ventas = spark.read.csv(
    path=path_ventas, header=True,
    schema="id INT, fecha DATE, producto STRING, cantidad INT, precio DOUBLE",
    dateFormat="yyyy-MM-dd", mode="FAILFAST"
)

display(df_ventas.limit(10))
df_ventas.printSchema()
print("Registros de venta:", df_ventas.count())

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


root
 |-- id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

Registros de venta: 100


## 7.1 Revisión antes de guardar
Buscamos llaves repetidas, nulos y valores no válidos para las operaciones del ejercicio. No esperamos filas en estas dos salidas. Si aparecen, revisamos el archivo antes de continuar.



In [0]:
display(df_ventas.groupBy("id").count().filter("count > 1"))

df_invalidos = df_ventas.filter(
    F.col("id").isNull() | F.col("fecha").isNull() |
    F.col("producto").isNull() | F.col("cantidad").isNull() |
    F.col("precio").isNull() | F.isnan("precio") |
    (F.col("cantidad") <= 0) | (F.col("precio") < 0)
)
display(df_invalidos)
assert df_invalidos.count() == 0, "Revisar los registros antes de continuar."

id,count


id,fecha,producto,cantidad,precio


## 7.2 Tabla de Ventas 
Guardamos la tabla base. En las siguientes operaciones modificaremos una copia de ensayo. La tabla ventas conservará los cien registros del CSV.



In [0]:
df_ventas.write.format("delta").mode("overwrite").saveAsTable("ventas")


In [0]:
%sql
DESCRIBE DETAIL ventas


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8ccc687a-e08e-4392-ba47-49def130dbd8,analitica.data.ventas,null,,2026-10-17T20:06:23.892Z,2026-10-17T20:06:26.000Z,List(),List(),1,2392,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 1f2c15b7-dd98-44e0-acf9-8e5d6f506506)",3,7,"List(appendOnly, deletionVectors, invariants, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## 7.3 Consulta del importe por producto
La consulta multiplica cantidad por precio y suma esos importes por producto. ROUND se aplica al total para mostrar dos decimales.



In [0]:
%sql
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto
ORDER BY importe_total DESC

producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


Databricks visualization. Run in Databricks to view.

# 8. Cambios y ensayos 
##8.1 Una variable para cambiar el filtro
Volvemos a los pedidos de Spark. Cambiamos pais_elegido por Spain, France o Germany y ejecutamos de nuevo la celda. El filtro no modifica la tabla guardada.

In [0]:
pais_elegido = "France"

display(df_enriquecido.filter(F.col("country") == pais_elegido))

product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,3,2024-01-05,France,Electronics,1,1150.0,1150.0,Computers,TechCorp
Mouse,6,2024-01-08,France,Accessories,10,23.0,230.0,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280.0,280.0,Displays,ScreenWorld


## 8.2 Copia de ensayo para Delta
La tabla ventas_delta_demo es independiente de ventas. CREATE OR REPLACE permite reiniciar sus datos con los cien registros originales antes de repetir la secuencia.



In [0]:
%sql
CREATE OR REPLACE TABLE ventas_delta_demo USING DELTA AS SELECT * FROM ventas


num_affected_rows,num_inserted_rows


## 8.3 Versión de referencia
Guardamos la versión actual antes de modificar filas. Al repetir el ejercicio puede tener un número diferente.

In [0]:
historial = spark.sql("DESCRIBE HISTORY ventas_delta_demo")
version_inicial = historial.orderBy(F.desc("version")).first()["version"]

print("Versión de referencia:", version_inicial)

Versión de referencia: 0


## 8.4 UPDATE
Aumentamos una unidad el precio del registro id = 1 en la copia de ensayo. Su precio original es 25,99 y después de este cambio será 26,99.

In [0]:
%sql
UPDATE ventas_delta_demo SET precio = ROUND(precio + 1, 2) WHERE id = 1


num_affected_rows
1


In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id = 1


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,26.99


## 8.5 INSERT
Insertamos una venta ficticia con id = 1001. La tabla pasa de cien a ciento una filas. Ejecutamos la inserción una sola vez dentro de esta secuencia.

In [0]:
%sql
INSERT INTO ventas_delta_demo VALUES (1001, DATE '2025-01-31', 'Producto demo', 2, 10.0)


num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo


registros
101


## 8.6 DELETE
Eliminamos únicamente la venta ficticia. La tabla vuelve a tener cien filas.

In [0]:
%sql
DELETE FROM ventas_delta_demo WHERE id = 1001


num_affected_rows
1


In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo


registros
100


## 8.7 Fuente de cambios para MERGE
La vista temporal contiene dos filas: id = 1 ya existe y llegará con precio 27,99. id = 1001 es nuevo. La fuente utiliza los mismos nombres de columna que el destino.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cambios_ventas AS
SELECT * FROM VALUES
  (1, DATE '2025-01-03', 'Teclado', 20, CAST(27.99 AS DOUBLE)),
  (1001, DATE '2025-01-31', 'Producto demo', 2, CAST(10.0 AS DOUBLE))
AS cambios(id, fecha, producto, cantidad, precio)

In [0]:
%sql
SELECT * FROM cambios_ventas


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


## 8.8 MERGE
Comparamos por id. Si la llave existe, actualizamos la fila. Si no existe, la insertamos. La fuente debe tener una sola fila por llave. 

In [0]:
%sql
MERGE INTO ventas_delta_demo AS destino
USING cambios_ventas AS fuente
ON destino.id = fuente.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id IN (1, 1001) ORDER BY id


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


# 9. Historial
Revisamos las operaciones y las versiones que produjeron.

In [0]:
%sql
DESCRIBE HISTORY ventas_delta_demo


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-17T20:41:47.000Z,76471736688769,carmenmarin689@gmail.com,MERGE,"Map(predicate -> [""(id#14498 = id#14483)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(431157744805419),9ba7e82e-b7dc-4484-b5fb-dc0b3d4e8321,1017-185820-he9ttdly-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3293, numTargetBytesRemoved -> 1660, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3958, materializeSourceTimeMs -> 306, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1883, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1674)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
3,2026-10-17T20:40:11.000Z,76471736688769,carmenmarin689@gmail.com,DELETE,"Map(predicate -> [""(id#14187 = 1001)""])",null,List(431157744805419),a230eb43-c9e0-4afd-bab0-cfd8da467d3d,1017-185820-he9ttdly-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1663, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1220, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1076, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 143)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-17T20:39:13.000Z,76471736688769,carmenmarin689@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(431157744805419),ae7fe954-a5c2-453e-86e5-e77108961a14,1017-185820-he9ttdly-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1663)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-17T20:38:31.000Z,76471736688769,carmenmarin689@gmail.com,UPDATE,"Map(predicate -> [""(id#13385 = 1)""])",null,List(431157744805419),53a4441e-d7ed-40b3-b127-0994e4b4d39e,1017-185820-he9ttdly-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4115, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2089, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1660, rewriteTimeMs -> 1976)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-17T20:17:42.000Z,76471736688769,carmenmarin689@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""b355638f-ac9e-404f-a0f5-724bdf3dead4""}, statsOnLoad -> true)",null,List(431157744805419),a4c2317d-4d97-4809-acc2-8045fd8580b7,1017-185820-he9ttdly-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2392)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


## 9.1 Time Travel
Leemos la versión registrada antes de los cambios y la comparamos con la actual. La versión de referencia tiene cien filas y precio 25,99 para id = 1. La actual tiene ciento una filas y precio 27,99. La lectura histórica necesita conservar los archivos y el registro de esa versión.

In [0]:
df_inicial = spark.read.option("versionAsOf", version_inicial).table("ventas_delta_demo")

print("Filas iniciales:", df_inicial.count())
print("Filas actuales:", spark.table("ventas_delta_demo").count())
display(df_inicial.filter(F.col("id") == 1))
display(spark.table("ventas_delta_demo").filter(F.col("id") == 1))


Filas iniciales: 100
Filas actuales: 101


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99


# COMPROBACIÓN

In [0]:
%sql
SELECT 'Pedidos enriquecidos' AS tabla, COUNT(*) AS registros FROM pedidos_enriquecidos
UNION ALL
SELECT 'Ventas base', COUNT(*) FROM ventas
UNION ALL
SELECT 'Ensayo Delta', COUNT(*) FROM ventas_delta_demo

tabla,registros
Pedidos enriquecidos,10
Ventas base,100
Ensayo Delta,101


# 10. Vistas, funciones y control de acceso
Trabajamos sobre ventas, que conserva el CSV original, para publicar una consulta reutilizable. La copia de ensayo permanece separada.

## 10.1 Creación de una vista
La vista guarda la consulta. Al consultarla obtenemos el resumen calculado desde su tabla base.

In [0]:
%sql
CREATE OR REPLACE VIEW v_ventas_por_producto AS
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto

In [0]:
%sql
SELECT * FROM v_ventas_por_producto ORDER BY importe_total DESC


producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


## 10.2 Función reutilizable
Definimos una regla de demostración: un importe supera el umbral si es mayor que 500. El umbral no es una clasificación oficial del negocio. Un valor ausente produce NULL en esta comparación.

In [0]:
%sql
CREATE OR REPLACE FUNCTION es_venta_alta(importe DOUBLE)
RETURNS BOOLEAN
RETURN importe > 500

In [0]:
%sql
SELECT id, producto, cantidad, precio,
       es_venta_alta(cantidad * precio) AS supera_umbral
FROM ventas
ORDER BY id
LIMIT 10

id,producto,cantidad,precio,supera_umbral
1,Teclado,20,25.99,true
2,Webcam,1,39.99,false
3,Auriculares,18,29.99,true
4,Webcam,10,39.99,false
5,Mouse,2,15.5,false
6,Altavoces,13,45.0,true
7,Teclado,11,25.99,false
8,Auriculares,5,29.99,false
9,Monitor,2,199.99,false
10,Microfono,20,59.99,true


## 10.3 Propiedad y permisos
Revisamos el propietario de la vista y las concesiones que nuestro usuario puede ver. Estos resultados describen el entorno en el que ejecutamos la práctica.

In [0]:
%sql
DESCRIBE TABLE EXTENDED v_ventas_por_producto


col_name,data_type,comment
producto,string,null
unidades,bigint,null
importe_total,double,null
,,
# Detailed Table Information,,
Catalog,analitica,
Database,data,
Table,v_ventas_por_producto,
Owner,carmenmarin689@gmail.com,
Created Time,Sat Oct 17 20:45:02 UTC 2026,


In [0]:
%sql
SHOW GRANTS ON VIEW v_ventas_por_producto


Principal,ActionType,ObjectType,ObjectKey


## 10.4 Concesión de acceso
El ejemplo utiliza lectores_clase. Antes de habilitarlo, el administrador debe crear el grupo y agregar a la persona que hará la prueba. Si el grupo tiene otro nombre, lo reemplazamos. Quitamos los dos guiones de cada concesión solo cuando corresponda aplicarla y tengamos permiso para hacerlo. Si solo estamos revisando la sintaxis, las dejamos comentadas.

In [0]:
%sql
-- GRANT USE CATALOG ON CATALOG analitica TO `lectores_clase`;
-- GRANT USE SCHEMA ON SCHEMA analitica.data TO `lectores_clase`;
-- GRANT SELECT ON VIEW analitica.data.v_ventas_por_producto TO `lectores_clase`;

SHOW GRANTS ON VIEW v_ventas_por_producto;

Principal,ActionType,ObjectType,ObjectKey


### RESPUESTAS 
**1. Si el CSV está en un volumen, ¿qué falta para que otro notebook consulte una tabla persistente?**
Falta convertirlo en tabla. Hay que leer el archivo con Spark y guardarlo como tabla Delta, como hice con data_demo.csv, que quedó guardado como inmuebles. Así queda registrado en el catálogo y otro notebook lo puede consultar por su nombre.

**2. Si una unión duplica el importe total, ¿qué comprobación haríamos sobre las llaves?**
Revisaría que cada producto aparezca una sola vez en la dimensión de productos. Si alguno está repetido, sus pedidos se duplican al unir. También compararía el número de filas antes y después de la unión.

**3. Si una celda define un filtro, ¿significa que Spark ya calculó todo el resultado?**
No. Spark solo arma el plan y lo ejecuta cuando se pide un resultado, por ejemplo al mostrar, contar o guardar los datos.

**4. Si una persona tiene SELECT sobre una tabla, ¿qué otros permisos de uso necesita?**
Necesita USE CATALOG sobre el catálogo y USE SCHEMA sobre el esquema. Sin ellos no puede llegar a la tabla.

**5. Si el historial de la tabla muestra una versión antigua, ¿basta para garantizar su recuperación?**
No. El historial solo muestra que la versión existió. Para recuperarla todavía deben existir sus archivos, y eso depende de la retención de la tabla.

**6. Si repetimos un append con el mismo archivo, ¿qué puede pasar con las ventas?**
Se duplican y los totales salen inflados. Pasa igual con la tabla simple: si el INSERT se ejecuta dos veces, quedan seis filas en vez de tres.